In [ ]:
import streamlit as st
from PyPDF2 import PdfReader
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# ---------------------------
# Extract text from PDF
# ---------------------------
def extract_text_from_pdf(file):
    pdf = PdfReader(file)
    text = ""
    for page in pdf.pages:
        text += page.extract_text()
    return text.lower()

# ---------------------------
# Extract keywords (simple)
# ---------------------------
def extract_keywords(text):
    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    return set(words)

# ---------------------------
# Rank resumes
# ---------------------------
def rank_resumes(job_description, resumes):
    documents = [job_description] + resumes
    vectorizer = TfidfVectorizer().fit_transform(documents)
    vectors = vectorizer.toarray()

    job_vector = vectors[0]
    resume_vectors = vectors[1:]

    similarities = cosine_similarity([job_vector], resume_vectors).flatten()
    return similarities * 100

# ---------------------------
# Analyze candidate
# ---------------------------
def analyze_candidate(job_desc, resume_text, score):
    jd_keywords = extract_keywords(job_desc)
    resume_keywords = extract_keywords(resume_text)

    matched = jd_keywords.intersection(resume_keywords)
    missing = jd_keywords.difference(resume_keywords)

    strengths = list(matched)[:3]
    gaps = list(missing)[:3]

    # Recommendation logic
    if score >= 75:
        recommendation = "Strong Fit"
    elif score >= 50:
        recommendation = "Moderate Fit"
    else:
        recommendation = "Not Fit"

    return strengths, gaps, recommendation

# ---------------------------
# Streamlit UI
# ---------------------------
st.title("AI Resume Screening & Candidate Ranking System")

st.header("Job Description")
job_description = st.text_area("Enter the job description")

st.header("Upload Resumes")
uploaded_files = st.file_uploader("Upload PDF files", type=["pdf"], accept_multiple_files=True)

if uploaded_files and job_description:
    st.header("Results")

    resumes = []
    names = []

    for file in uploaded_files:
        text = extract_text_from_pdf(file)
        resumes.append(text)
        names.append(file.name)

    scores = rank_resumes(job_description, resumes)

    results = []

    for i in range(len(resumes)):
        strengths, gaps, recommendation = analyze_candidate(
            job_description, resumes[i], scores[i]
        )

        results.append({
            "Candidate": names[i],
            "Match Score (0–100)": round(scores[i], 2),
            "Key Strengths": ", ".join(strengths),
            "Key Gaps": ", ".join(gaps),
            "Final Recommendation": recommendation
        })

    df = pd.DataFrame(results)
    df = df.sort_values(by="Match Score (0–100)", ascending=False)

    # Add ranking
    df["Rank"] = range(1, len(df) + 1)

    st.dataframe(df)

    # 🔥 Fancy display per candidate
    for i, row in df.iterrows():
        st.subheader(f"🏆 Rank {row['Rank']} - {row['Candidate']}")
        st.write(f"**Match Score:** {row['Match Score (0–100)']}%")
        st.write(f"**Strengths:** {row['Key Strengths']}")
        st.write(f"**Gaps:** {row['Key Gaps']}")
        st.write(f"**Recommendation:** {row['Final Recommendation']}")
        st.progress(int(row["Match Score (0–100)"]))
